# Parametric NAM Training (Colab)

Put this notebook at `MyDrive/parametric-training/train_parametric_colab.ipynb` and open it in Google Colab.

This notebook:
- mounts Google Drive,
- clones or unpacks your branch,
- treats `data.json`, `model.json`, and `learning.json` as the source of truth,
- writes runtime-only config copies for Colab so your originals stay untouched,
- runs `nam-full`,
- leaves all outputs in Drive so you can compare runs later.

If you reuse configs exported by the GUI trainer, the notebook only applies small Colab-specific overrides like log-friendly trainer settings and Drive path resolution.


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil
import sys


drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/parametric-training')
CONFIG_SOURCE_DIR = DRIVE_ROOT / 'low_high_outputs_v3'
RUNTIME_CONFIG_DIR = Path('/content/parametric_runtime_configs')
RUNS_ROOT = DRIVE_ROOT / 'runs' / CONFIG_SOURCE_DIR.name
REPO_DIR = Path('/content/nam_parametric')

RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'DRIVE_ROOT      = {DRIVE_ROOT}')
print(f'CONFIG_SOURCE   = {CONFIG_SOURCE_DIR}')
print(f'RUNTIME_CONFIGS = {RUNTIME_CONFIG_DIR}')
print(f'RUNS_ROOT       = {RUNS_ROOT}')
print(f'REPO_DIR        = {REPO_DIR}')


## Repo Setup

Choose one source mode:
- `git`: clone your pushed branch or commit from GitHub.
- `zip`: upload a repo zip to `MyDrive/parametric-training/nam_parametric_repo.zip` if your local changes are not pushed yet.


In [ ]:
SOURCE_MODE = 'git'  # change to 'zip' if needed
REPO_URL = 'https://github.com/phillipmself/neural-amp-modeler-parametric.git'
REPO_REF = 'feature/nam-a2-parametric'
REPO_ZIP = DRIVE_ROOT / 'nam_parametric_repo.zip'

import os
import re
import subprocess

os.chdir('/content')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if SOURCE_MODE == 'git':
    if 'YOUR-USERNAME' in REPO_URL or REPO_REF == 'YOUR-BRANCH-OR-COMMIT':
        raise ValueError('Set REPO_URL and REPO_REF before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'checkout', REPO_REF], check=True)
elif SOURCE_MODE == 'zip':
    if not REPO_ZIP.exists():
        raise FileNotFoundError(f'Repo zip not found: {REPO_ZIP}')
    extract_root = Path('/content/repo_extract')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(REPO_ZIP), str(extract_root))
    repo_candidates = [
        p
        for p in extract_root.iterdir()
        if p.is_dir() and p.name != '__MACOSX' and not p.name.startswith('.')
    ]
    repo_marker_names = ('pyproject.toml', 'setup.py', '.git')
    extract_root_looks_like_repo = any((extract_root / name).exists() for name in repo_marker_names)
    if len(repo_candidates) == 1:
        shutil.move(str(repo_candidates[0]), str(REPO_DIR))
    elif len(repo_candidates) == 0 and extract_root_looks_like_repo:
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        for child in extract_root.iterdir():
            if child.name == '__MACOSX':
                continue
            shutil.move(str(child), str(REPO_DIR / child.name))
    else:
        raise RuntimeError(
            'Expected one repo directory in zip after filtering archive metadata, '
            f'found directories: {repo_candidates}'
        )
    os.chdir(REPO_DIR)
else:
    raise ValueError(f'Unsupported SOURCE_MODE: {SOURCE_MODE}')

print('Repo ready at', REPO_DIR)
print('Current working directory:', Path.cwd())


In [ ]:
import os
import subprocess

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel', 'tensorboard'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)], check=True)

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not detected. In Colab, switch Runtime -> Change runtime type -> T4/L4/A100 GPU.')


In [ ]:
from copy import deepcopy
from datetime import datetime
import json
import os
import re
import subprocess
import sys
import time

from IPython.display import clear_output

RUNS_BASE = DRIVE_ROOT / 'runs'
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0


def load_json(path: Path):
    with path.open('r') as fp:
        return json.load(fp)


def save_json(path: Path, payload):
    with path.open('w') as fp:
        json.dump(payload, fp, indent=4)
        fp.write('\n')


def as_entries(entries):
    if entries is None:
        return []
    if isinstance(entries, dict):
        return [entries]
    if isinstance(entries, list):
        return entries
    raise TypeError(f'Unsupported split type: {type(entries)}')


def deep_merge_dict(base: dict, overrides: dict):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_merge_dict(base[key], value)
        else:
            base[key] = deepcopy(value)
    return base


def resolve_config_dir(raw_path) -> Path:
    path = Path(raw_path).expanduser()
    if path.is_absolute():
        return path
    drive_candidate = DRIVE_ROOT / path
    if drive_candidate.exists():
        return drive_candidate
    return path.resolve()


def resolve_audio_path(raw_path: str, *, config_dir: Path, drive_root: Path) -> Path:
    raw = Path(raw_path)
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            config_dir / raw,
            drive_root / raw,
            raw,
        ])
    if raw.name:
        candidates.extend([
            config_dir / raw.name,
            drive_root / raw.name,
        ])

    deduped = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(candidate)

    for candidate in deduped:
        if candidate.exists():
            return candidate

    if raw.name:
        return config_dir / raw.name
    return config_dir / raw


def normalize_capture_entries(
    data_config: dict,
    split_name: str,
    *,
    config_dir: Path,
    drive_root: Path,
    notes: list[str],
):
    original_entries = data_config.get(split_name)
    if original_entries is None:
        return

    is_single_entry = isinstance(original_entries, dict)
    entries = deepcopy(as_entries(original_entries))
    for entry in entries:
        original_path = entry['y_path']
        resolved_path = resolve_audio_path(
            original_path,
            config_dir=config_dir,
            drive_root=drive_root,
        )
        if str(resolved_path) != original_path:
            notes.append(
                f"Resolved {split_name} capture {Path(original_path).name} -> {resolved_path}"
            )
        entry['y_path'] = str(resolved_path)
        entry['delay'] = int(entry.get('delay', 0) or 0)

    data_config[split_name] = entries[0] if is_single_entry else entries


def prepare_runtime_configs(config_dir: Path, drive_root: Path):
    data = load_json(config_dir / 'data.json')
    model = load_json(config_dir / 'model.json')
    learning = load_json(config_dir / 'learning.json')

    if model.get('net', {}).get('name') != 'ParametricWaveNet':
        raise ValueError(
            "This notebook expects model.json to use net.name='ParametricWaveNet'."
        )

    runtime_data = deepcopy(data)
    runtime_model = deepcopy(model)
    runtime_learning = deepcopy(learning)
    notes = []

    common = runtime_data.setdefault('common', {})
    if 'x_path' not in common:
        raise KeyError("data.json is missing common.x_path")

    original_input_path = common['x_path']
    resolved_input_path = resolve_audio_path(
        original_input_path,
        config_dir=config_dir,
        drive_root=drive_root,
    )
    if str(resolved_input_path) != original_input_path:
        notes.append(
            f"Resolved input capture {Path(original_input_path).name} -> {resolved_input_path}"
        )
    common['x_path'] = str(resolved_input_path)
    common['allow_unequal_lengths'] = True
    common['require_input_pre_silence'] = None

    for split_name in ('train', 'validation', 'verification'):
        normalize_capture_entries(
            runtime_data,
            split_name,
            config_dir=config_dir,
            drive_root=drive_root,
            notes=notes,
        )

    trainer = runtime_learning.setdefault('trainer', {})
    trainer.setdefault('accelerator', 'gpu')
    trainer.setdefault('devices', 1)
    trainer.setdefault('deterministic', 'warn')
    trainer.setdefault('max_epochs', 100)

    accelerator = trainer.get('accelerator')
    if trainer.get('deterministic') is True and accelerator in ('gpu', 'cuda'):
        trainer['deterministic'] = 'warn'
        notes.append(
            "Changed trainer.deterministic from True to 'warn' for CUDA compatibility with reflection_pad1d backward."
        )

    runtime_overrides = {
        'enable_progress_bar': True,
        'enable_model_summary': True,
        'logger': True,
        'log_every_n_steps': 1,
    }
    for key, value in runtime_overrides.items():
        if trainer.get(key) != value:
            notes.append(f"Set trainer.{key} = {value!r} for notebook runtime")
            trainer[key] = value

    runtime_learning.setdefault('trainer_fit_kwargs', {})

    has_verification = (
        'verification' in runtime_data and 'verification_windows' in runtime_data
    )
    if has_verification and 'checkpoint_monitor' not in runtime_learning:
        runtime_learning['checkpoint_monitor'] = 'val_loss_unseen_audio_unseen_param'
        notes.append(
            'Set checkpoint_monitor to val_loss_unseen_audio_unseen_param for held-out verification captures'
        )

    return runtime_data, runtime_model, runtime_learning, notes


def collect_required_paths(data_config: dict, config_dir: Path):
    paths = [
        config_dir / 'data.json',
        config_dir / 'model.json',
        config_dir / 'learning.json',
        Path(data_config['common']['x_path']),
    ]
    for split_name in ('train', 'validation', 'verification'):
        for entry in as_entries(data_config.get(split_name)):
            paths.append(Path(entry['y_path']))

    ordered = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        ordered.append(path)
    return ordered


def sanitize_name(value: str) -> str:
    sanitized = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '-' for ch in value)
    sanitized = sanitized.strip('-_')
    return sanitized or 'run'


def run_timestamp() -> str:
    return datetime.now().strftime('%Y-%m-%d-%H-%M-%S-%f')


def make_run_label(config_dir: Path, run_name: str | None) -> str:
    base_name = sanitize_name(run_name or config_dir.name)
    return f"{base_name}-{run_timestamp()}"


def prepare_run_bundle(spec: dict):
    spec = deepcopy(spec)
    config_dir = resolve_config_dir(spec.get('config_dir', CONFIG_SOURCE_DIR))
    if not config_dir.exists():
        raise FileNotFoundError(f'Config directory not found: {config_dir}')

    data_config, model_config, learning_config, runtime_notes = prepare_runtime_configs(
        config_dir,
        DRIVE_ROOT,
    )

    for payload_name, payload, override_key in (
        ('data', data_config, 'data_overrides'),
        ('model', model_config, 'model_overrides'),
        ('learning', learning_config, 'learning_overrides'),
    ):
        overrides = spec.get(override_key)
        if overrides:
            deep_merge_dict(payload, overrides)
            runtime_notes.append(
                f'Applied {payload_name} overrides from run spec key {override_key}.'
            )

    run_label = make_run_label(config_dir, spec.get('run_name'))
    runtime_dir = RUNTIME_CONFIG_DIR / run_label
    run_root = RUNS_BASE / config_dir.name
    run_outdir = run_root / run_label

    runtime_dir.mkdir(parents=True, exist_ok=False)
    run_outdir.mkdir(parents=True, exist_ok=False)
    (run_outdir / 'lightning_logs').mkdir(parents=True, exist_ok=True)

    runtime_paths = {
        'data': runtime_dir / 'data.json',
        'model': runtime_dir / 'model.json',
        'learning': runtime_dir / 'learning.json',
    }
    save_json(runtime_paths['data'], data_config)
    save_json(runtime_paths['model'], model_config)
    save_json(runtime_paths['learning'], learning_config)

    return {
        'spec': spec,
        'config_dir': config_dir,
        'config_name': config_dir.name,
        'run_label': run_label,
        'runtime_dir': runtime_dir,
        'runtime_paths': runtime_paths,
        'run_outdir': run_outdir,
        'log_path': run_outdir / 'train.log',
        'gpu': spec.get('gpu'),
        'data_config': data_config,
        'model_config': model_config,
        'learning_config': learning_config,
        'runtime_notes': runtime_notes,
    }


def write_launcher_script() -> Path:
    launcher_path = Path('/content/nam_parametric_notebook_launcher.py')
    launcher_path.write_text(
        "\n".join(
            [
                'import argparse',
                'import json',
                'from contextlib import suppress',
                'from pathlib import Path',
                '',
                'import nam.train.full as nam_full_module',
                'from lightning.pytorch.callbacks import Callback',
                '',
                'with suppress(ModuleNotFoundError, ImportError):',
                '    from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger',
                '',
                'def load_json(path: str):',
                '    with open(path, \'r\') as fp:',
                '        return json.load(fp)',
                '',
                'class NotebookProgressCallback(Callback):',
                '    def __init__(self, progress_path: Path):',
                '        super().__init__()',
                '        self._progress_path = progress_path',
                '        self._last_saved_global_step = None',
                '',
                '    def _write(self, trainer, *, status: str):',
                '        payload = {',
                '            \"status\": status,',
                '            \"current_epoch\": int(trainer.current_epoch) + 1,',
                '            \"global_step\": int(trainer.global_step),',
                '            \"max_epochs\": int(trainer.max_epochs),',
                '        }',
                '        self._progress_path.write_text(json.dumps(payload))',
                '',
                '    def on_fit_start(self, trainer, pl_module):',
                '        self._write(trainer, status=\"starting\")',
                '',
                '    def on_train_epoch_start(self, trainer, pl_module):',
                '        self._write(trainer, status=\"training\")',
                '',
                '    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):',
                '        global_step = int(trainer.global_step)',
                '        if global_step == 0 or global_step == self._last_saved_global_step:',
                '            return',
                '        if global_step % 25 == 0:',
                '            self._write(trainer, status=\"training\")',
                '            self._last_saved_global_step = global_step',
                '',
                '    def on_validation_end(self, trainer, pl_module):',
                '        if getattr(trainer, \"sanity_checking\", False):',
                '            return',
                '        self._write(trainer, status=\"validating\")',
                '',
                '    def on_fit_end(self, trainer, pl_module):',
                '        self._write(trainer, status=\"finished\")',
                '',
                'def build_notebook_loggers(outdir: Path):',
                '    if \"TensorBoardLogger\" not in globals():',
                '        return True',
                '    return [',
                '        TensorBoardLogger(save_dir=str(outdir), name=\"lightning_logs\"),',
                '        CSVLogger(save_dir=str(outdir), name=\"csv_logs\"),',
                '    ]',
                '',
                'OriginalTrainer = nam_full_module._pl.Trainer',
                '',
                'def notebook_trainer_factory(*args, **kwargs):',
                '    outdir = Path(kwargs.get(\"default_root_dir\", \".\"))',
                '    callbacks = list(kwargs.get(\"callbacks\") or [])',
                '    callbacks.append(NotebookProgressCallback(outdir / \"notebook_progress.json\"))',
                '    kwargs[\"callbacks\"] = callbacks',
                '    if kwargs.get(\"logger\", True) is True:',
                '        kwargs[\"logger\"] = build_notebook_loggers(outdir)',
                '    return OriginalTrainer(*args, **kwargs)',
                '',
                'nam_full_module._pl.Trainer = notebook_trainer_factory',
                '',
                'parser = argparse.ArgumentParser()',
                'parser.add_argument(\'--data\', required=True)',
                'parser.add_argument(\'--model\', required=True)',
                'parser.add_argument(\'--learning\', required=True)',
                'parser.add_argument(\'--outdir\', required=True)',
                'args = parser.parse_args()',
                '',
                'data_config = load_json(args.data)',
                'model_config = load_json(args.model)',
                'learning_config = load_json(args.learning)',
                'outdir = Path(args.outdir)',
                'outdir.mkdir(parents=True, exist_ok=True)',
                '',
                'nam_full_module.main(',
                '    data_config,',
                '    model_config,',
                '    learning_config,',
                '    outdir,',
                '    no_show=True,',
                '    make_plots=False,',
                ')',
                '',
            ]
        )
    )
    return launcher_path


def resource_key_for_bundle(bundle: dict):
    trainer = bundle['learning_config'].get('trainer', {})
    accelerator = trainer.get('accelerator')
    if accelerator not in ('gpu', 'cuda'):
        return None
    if bundle['gpu'] is None:
        return 'default-gpu'
    return f"gpu:{bundle['gpu']}"


def launch_run_process(bundle: dict, launcher_path: Path):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if bundle['gpu'] is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(bundle['gpu'])

    command = [
        sys.executable,
        '-u',
        str(launcher_path),
        '--data',
        str(bundle['runtime_paths']['data']),
        '--model',
        str(bundle['runtime_paths']['model']),
        '--learning',
        str(bundle['runtime_paths']['learning']),
        '--outdir',
        str(bundle['run_outdir']),
    ]

    log_handle = bundle['log_path'].open('w')
    process = subprocess.Popen(
        command,
        cwd=str(REPO_DIR),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    return {
        **bundle,
        'command': command,
        'process': process,
        'log_handle': log_handle,
        'pid': process.pid,
        'started_at': time.time(),
        'resource_key': resource_key_for_bundle(bundle),
    }


def query_gpu_snapshot():
    try:
        result = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=index,name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu',
                '--format=csv,noheader,nounits',
            ],
            capture_output=True,
            text=True,
            check=False,
        )
    except FileNotFoundError:
        return ['nvidia-smi not available in this runtime.']

    if result.returncode != 0:
        message = (result.stderr or result.stdout or 'Unknown nvidia-smi error').strip()
        return [f'nvidia-smi failed: {message}']

    rows = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    if not rows:
        return ['No GPU rows returned by nvidia-smi.']

    summary = []
    for row in rows:
        fields = [field.strip() for field in row.split(',')]
        if len(fields) < 7:
            summary.append(row)
            continue
        idx, name, util_gpu, util_mem, mem_used, mem_total, temp = fields[:7]
        summary.append(
            f'GPU {idx} {name}: util {util_gpu}% | mem {mem_used}/{mem_total} MB | mem util {util_mem}% | temp {temp}C'
        )
    return summary


def tail_text(path: Path, lines: int = 12):
    if not path.exists():
        return ['<log file not created yet>']
    text = path.read_text(errors='replace').replace('\r', '\n')
    tail = text.splitlines()[-lines:]
    return tail or ['<log file is empty>']


def load_notebook_progress(run_outdir: Path):
    progress_path = run_outdir / 'notebook_progress.json'
    if not progress_path.exists():
        return None
    try:
        return json.loads(progress_path.read_text())
    except json.JSONDecodeError:
        return None


def latest_epoch_from_log(path: Path):
    if not path.exists():
        return None
    text = path.read_text(errors='replace')
    matches = re.findall(r'Epoch\s+(\d+):', text)
    if not matches:
        return None
    return int(matches[-1]) + 1


def latest_completed_epoch_from_checkpoints(run_outdir: Path):
    checkpoint_dir = run_outdir / 'checkpoints'
    if not checkpoint_dir.exists():
        return None

    epochs = []
    for checkpoint_path in checkpoint_dir.glob('checkpoint_epoch_*.ckpt'):
        suffix = checkpoint_path.stem.rsplit('_', 1)[-1]
        if suffix.isdigit():
            epochs.append(int(suffix) + 1)
    return max(epochs) if epochs else None


def describe_epoch_progress(info: dict) -> str:
    max_epochs = info.get('learning_config', {}).get('trainer', {}).get('max_epochs')
    progress = load_notebook_progress(info['run_outdir'])
    if isinstance(progress, dict):
        progress_epoch = progress.get('current_epoch')
        progress_step = progress.get('global_step')
        progress_max_epochs = progress.get('max_epochs')
        progress_status = progress.get('status', 'training')
        if isinstance(progress_max_epochs, int) and progress_max_epochs > 0:
            max_epochs = progress_max_epochs
        if isinstance(progress_epoch, int) and progress_epoch > 0:
            if isinstance(max_epochs, int) and max_epochs > 0:
                return f'epoch: {progress_epoch}/{max_epochs} | step: {progress_step} | {progress_status}'
            return f'epoch: {progress_epoch} | step: {progress_step} | {progress_status}'

    live_epoch = latest_epoch_from_log(info['log_path'])
    completed_epoch = latest_completed_epoch_from_checkpoints(info['run_outdir'])

    if live_epoch is not None:
        current_epoch = live_epoch
        source = 'live log'
    elif completed_epoch is not None:
        current_epoch = completed_epoch
        source = 'epoch checkpoint'
    else:
        return 'epoch: starting...'

    if isinstance(max_epochs, int) and max_epochs > 0:
        return f'epoch: {current_epoch}/{max_epochs} ({source})'
    return f'epoch: {current_epoch} ({source})'


def format_elapsed(started_at: float, ended_at: float | None = None) -> str:
    stop = time.time() if ended_at is None else ended_at
    total_seconds = int(max(stop - started_at, 0))
    minutes, seconds = divmod(total_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f'{hours}h {minutes}m {seconds}s'
    if minutes:
        return f'{minutes}m {seconds}s'
    return f'{seconds}s'


def render_monitor_view(
    *,
    pending: list[dict],
    running: list[dict],
    completed: list[dict],
    gpu_lines: list[str],
    log_tail_lines: int,
):
    clear_output(wait=True)
    print('Training monitor')
    print('=' * 80)
    print(f'Pending: {len(pending)} | Running: {len(running)} | Completed: {len(completed)}')
    print()
    print('GPU snapshot:')
    for line in gpu_lines:
        print('  ', line)

    if running:
        print()
        print('Active runs:')
        for info in running:
            gpu_label = 'default' if info['gpu'] is None else str(info['gpu'])
            print(
                f"- {info['run_label']} | pid={info['pid']} | gpu={gpu_label} | elapsed={format_elapsed(info['started_at'])}"
            )
            print(f"  {describe_epoch_progress(info)}")
            print(f"  outdir: {info['run_outdir']}")
            print(f"  log: {info['log_path']}")
            for line in tail_text(info['log_path'], lines=log_tail_lines):
                print(f'    {line}')

    if completed:
        print()
        print('Completed runs:')
        for info in completed[-5:]:
            status = 'OK' if info['returncode'] == 0 else f"FAILED ({info['returncode']})"
            print(
                f"- {info['run_label']} | {status} | elapsed={format_elapsed(info['started_at'], info['ended_at'])}"
            )
            print(f"  {describe_epoch_progress(info)}")
            print(f"  outdir: {info['run_outdir']}")


def next_launchable_index(
    pending: list[dict],
    running: list[dict],
    *,
    allow_gpu_oversubscription: bool,
):
    if allow_gpu_oversubscription:
        return 0 if pending else None

    busy_resources = {
        info['resource_key']
        for info in running
        if info.get('resource_key') is not None
    }
    for index, bundle in enumerate(pending):
        resource_key = resource_key_for_bundle(bundle)
        if resource_key is None or resource_key not in busy_resources:
            return index
    return None


def run_training_plan(
    run_bundles: list[dict],
    *,
    max_concurrent_runs: int,
    allow_gpu_oversubscription: bool,
    gpu_monitor_interval_seconds: int,
    refresh_interval_seconds: int,
    log_tail_lines: int,
):
    if not run_bundles:
        raise ValueError('run_bundles must not be empty')
    if max_concurrent_runs < 1:
        raise ValueError('max_concurrent_runs must be at least 1')

    launcher_path = write_launcher_script()
    pending = list(run_bundles)
    running = []
    completed = []
    gpu_lines = ['GPU snapshot will appear here after the first poll.']
    next_gpu_poll_at = 0.0

    while pending or running:
        while len(running) < max_concurrent_runs:
            next_index = next_launchable_index(
                pending,
                running,
                allow_gpu_oversubscription=allow_gpu_oversubscription,
            )
            if next_index is None:
                break
            info = launch_run_process(pending.pop(next_index), launcher_path)
            running.append(info)

        now = time.time()
        if now >= next_gpu_poll_at:
            gpu_lines = query_gpu_snapshot()
            next_gpu_poll_at = now + max(gpu_monitor_interval_seconds, 1)

        still_running = []
        for info in running:
            returncode = info['process'].poll()
            if returncode is None:
                still_running.append(info)
                continue
            info['returncode'] = returncode
            info['ended_at'] = time.time()
            info['log_handle'].close()
            completed.append(info)
        running = still_running

        render_monitor_view(
            pending=pending,
            running=running,
            completed=completed,
            gpu_lines=gpu_lines,
            log_tail_lines=log_tail_lines,
        )

        if pending or running:
            time.sleep(max(refresh_interval_seconds, 1))

    failures = [info for info in completed if info['returncode'] != 0]
    if failures:
        failed_labels = ', '.join(info['run_label'] for info in failures)
        raise RuntimeError(f'{len(failures)} run(s) failed: {failed_labels}')

    return completed


In [ ]:
RUN_SPECS = [
    {
        'config_dir': CONFIG_SOURCE_DIR,
        'run_name': 'baseline',
        # 'gpu': 0,
        # 'learning_overrides': {'trainer': {'max_epochs': 50}},
        # 'model_overrides': {},
        # 'data_overrides': {},
    },
    # {
    #     'config_dir': DRIVE_ROOT / 'another-config-dir',
    #     'run_name': 'variant-b',
    #     # 'gpu': 1,
    #     # 'learning_overrides': {'trainer': {'max_epochs': 50}},
    # },
]

MAX_CONCURRENT_RUNS = 2
ALLOW_GPU_OVERSUBSCRIPTION = True
GPU_MONITOR_INTERVAL_SECONDS = 5
REFRESH_INTERVAL_SECONDS = 2
LOG_TAIL_LINES = 10

prepared_runs = [prepare_run_bundle(spec) for spec in RUN_SPECS]

missing = sorted(
    {
        str(path)
        for bundle in prepared_runs
        for path in collect_required_paths(bundle['data_config'], bundle['config_dir'])
        if not path.exists()
    }
)
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

print('Prepared run plan:')
print(f'GPU count detected: {GPU_COUNT}')
print(f'Max concurrent runs: {MAX_CONCURRENT_RUNS}')
print(f'Allow GPU oversubscription: {ALLOW_GPU_OVERSUBSCRIPTION}')
print()

for bundle in prepared_runs:
    print(f"Run: {bundle['run_label']}")
    print(f"  config dir: {bundle['config_dir']}")
    print(f"  outdir: {bundle['run_outdir']}")
    print(f"  runtime dir: {bundle['runtime_dir']}")
    print(f"  gpu: {bundle['gpu'] if bundle['gpu'] is not None else 'default'}")
    if bundle['runtime_notes']:
        print('  runtime adjustments:')
        for note in bundle['runtime_notes']:
            print('   -', note)
    print()


In [ ]:
RECENT_RUNS = run_training_plan(
    prepared_runs,
    max_concurrent_runs=MAX_CONCURRENT_RUNS,
    allow_gpu_oversubscription=ALLOW_GPU_OVERSUBSCRIPTION,
    gpu_monitor_interval_seconds=GPU_MONITOR_INTERVAL_SECONDS,
    refresh_interval_seconds=REFRESH_INTERVAL_SECONDS,
    log_tail_lines=LOG_TAIL_LINES,
)


In [ ]:
if 'RECENT_RUNS' in globals() and RECENT_RUNS:
    run_dirs = [Path(info['run_outdir']) for info in RECENT_RUNS]
else:
    run_dirs = sorted(path for path in (DRIVE_ROOT / 'runs').rglob('*') if path.is_dir() and (path / 'lightning_logs').exists())

if not run_dirs:
    raise RuntimeError(f'No run directories with lightning_logs found under {DRIVE_ROOT / "runs"}')

print('Run outputs:')
for run_dir in run_dirs:
    print(f'\n{run_dir}')
    top_level = sorted(path.name for path in run_dir.iterdir())
    print('  files:', ', '.join(top_level))
    event_files = sorted(run_dir.rglob('events.out.tfevents*'))
    print('  tensorboard event files:')
    for path in event_files:
        print('   ', path)


In [ ]:
TB_LOGDIR = str(DRIVE_ROOT / 'runs')
%reload_ext tensorboard
%tensorboard --logdir {TB_LOGDIR} --reload_interval 5


In [ ]:
import csv

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

if 'RECENT_RUNS' in globals() and RECENT_RUNS:
    run_dir = Path(RECENT_RUNS[-1]['run_outdir'])
else:
    run_dirs = sorted(
        path
        for path in (DRIVE_ROOT / 'runs').rglob('*')
        if path.is_dir() and (path / 'lightning_logs').exists()
    )
    if not run_dirs:
        raise RuntimeError(f'No run directories with lightning_logs found under {DRIVE_ROOT / "runs"}')
    run_dir = run_dirs[-1]

event_files = sorted((run_dir / 'lightning_logs').rglob('events.out.tfevents*'))
if not event_files:
    raise RuntimeError(f"No event files found under {run_dir / 'lightning_logs'}")

rows = []
scalar_tags = set()

for event_file in event_files:
    ea = EventAccumulator(str(event_file))
    ea.Reload()
    tags = ea.Tags().get('scalars', [])
    for tag in tags:
        scalar_tags.add(tag)
        for event in ea.Scalars(tag):
            rows.append(
                {
                    'run': run_dir.name,
                    'event_file': event_file.name,
                    'tag': tag,
                    'step': event.step,
                    'wall_time': event.wall_time,
                    'value': event.value,
                }
            )

rows.sort(key=lambda row: (row['tag'], row['step'], row['wall_time']))

out_csv = run_dir / 'all_scalars.csv'
with out_csv.open('w', newline='') as fp:
    writer = csv.DictWriter(
        fp,
        fieldnames=['run', 'event_file', 'tag', 'step', 'wall_time', 'value'],
    )
    writer.writeheader()
    writer.writerows(rows)

print(f'Wrote {out_csv}')
print(f'Scalar tags: {sorted(scalar_tags)}')
print(f'Rows: {len(rows)}')
